In [1]:
# Provide the input dataset filename, model foldername and output response filenames
model_folder="ai-medical-model-gpu-1epoch-test"
test_data_file="ai_medical_test_dataset_10rows.csv"
response_filename='FTmodel_GPU_1epoch.csv'


In [2]:
import os
current_directory = os.getcwd()
# New model name
new_model="ai-medical-model-gpu-10epoch-test1" #"Medical-Mind-Llama-3-8b"
# Save the fine-tuned model
save_path = os.path.join(current_directory, "models", model_folder)
os.makedirs(save_path, exist_ok=True)
save_path

'/home/quest/Shyam/finetuning/models/ai-medical-model-gpu-1epoch-test'

In [3]:
## Inference
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
max_seq_length = 2048
dtype = None
load_in_4bit = True
fine_tuned_model = AutoModelForCausalLM.from_pretrained(save_path, load_in_4bit=load_in_4bit)
tokenizer = AutoTokenizer.from_pretrained(save_path)

The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.
`low_cpu_mem_usage` was None, now set to True since model is quantized.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [4]:
# Prepare the model for inference
fine_tuned_model.eval()

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaSdpaAttention(
          (q_proj): lora.Linear4bit(
            (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
            (lora_dropout): ModuleDict(
              (default): Dropout(p=0.1, inplace=False)
            )
            (lora_A): ModuleDict(
              (default): Linear(in_features=4096, out_features=16, bias=False)
            )
            (lora_B): ModuleDict(
              (default): Linear(in_features=16, out_features=4096, bias=False)
            )
            (lora_embedding_A): ParameterDict()
            (lora_embedding_B): ParameterDict()
          )
          (k_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (v_proj): lora.Linear4bit(
            (base_layer): Linear4bit(in_features=4096, out_features=1024, bias=False)
            (

In [5]:
from datasets import load_dataset
#Load the test dataset from local
dataset = load_dataset("csv", data_files=test_data_file,split="train", cache_dir="opt/ml/input")
dataset

Dataset({
    features: ['question', 'context'],
    num_rows: 10
})

In [6]:
localdataset_df = dataset.to_pandas()
localdataset_df.isnull().value_counts()

question  context
False     False      10
Name: count, dtype: int64

In [7]:
localdataset_df = localdataset_df.dropna()
localdataset_df.isnull().value_counts()

question  context
False     False      10
Name: count, dtype: int64

In [8]:
localdataset_df

,question,context
0,What is CTLA4-Ig Preserved?,CTLA4-Ig Preserves Thymus-Derived T Regulatory...
1,What is the result of an examination of campto...,An examination of camptocormia assessment by d...
2,What is the main form of treatment for psoriasis?,"In contrast to many other diseases, modern pso..."
3,What is the relationship between implicit self...,A growing body of work suggests that both depr...
4,What is CD147 known as?,CD147 or EMMPRIN is a member of the immunoglob...
5,What is the tyrphostin called?,"The tyrphostin, NT157, suppresses insulin rece..."
6,What is the only second row transition metal e...,Molybdenum is the only second row transition m...
7,What is a key component of the Enhanced Recove...,Nonsteroidal anti-inflammatory drug (NSAID) us...
8,What is the only opportunity to obtain diagnos...,Fine-needle aspiration has assumed an increasi...
9,What is the Onset of Arthritis in a Mouse Model?,Psoriatic Inflammation Facilitates the Onset o...


In [9]:
question=[]
for index, row in localdataset_df.iterrows():
    print(row['question'])
    #question.append(row['question'])
#question

What is CTLA4-Ig Preserved?
What is the result of an examination of camptocormia assessment?
What is the main form of treatment for psoriasis?
What is the relationship between implicit self-esteem and depression?
What is CD147 known as?
What is the tyrphostin called?
What is the only second row transition metal essential for biological systems?
What is a key component of the Enhanced Recovery After Surgery protocols?
What is the only opportunity to obtain diagnostic tissue to diagnose and subclassify RCC?
What is the Onset of Arthritis in a Mouse Model?


In [10]:
from datasets import Dataset, DatasetDict
test_ds = Dataset.from_pandas(localdataset_df)
test_ds

Dataset({
    features: ['question', 'context'],
    num_rows: 10
})

In [11]:
questions_list=[]
answers_list=[]
test_inference=True
if test_inference:
    for index, row in localdataset_df.iterrows():
      question=row['question']
      #question="What is the support for adult patients with acute respiratory failure?"
      prompt = f"<|start_header_id|>system<|end_header_id|> You are a Medical AI chatbot assistant. <|eot_id|><|start_header_id|>User: <|end_header_id|>{question}<|eot_id|>"
      # Tokenizing the input and generating the output
      #prompt = f"{question}"
      # Tokenizing the input and generating the output
      inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
      outputs = fine_tuned_model.generate(**inputs, max_new_tokens=256, use_cache=True)
      answer = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
      # Try Remove the prompt
      try:
          # Split the answer at the first line break, assuming system intro and question are on separate lines
          answer_parts = answer.split("\n", 1)
          # If there are multiple parts, consider the second part as the answer
          if len(answer_parts) > 1:
            answers = answer_parts[1].strip()  # Remove leading/trailing whitespaces
          else:
            answers = ""  # If no split possible, set answer to empty string
          print(f"Answer: {answers}") 
          answers_list.append(f"Answer: {answers}")  
      except:
          print(answer)
      questions_list.append(question)
      
#print(questions_list)
#print(answers_list)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
/home/quest/anaconda3/envs/yolov3_carla/lib/python3.10/site-packages/bitsandbytes/nn/modules.py:426: UserWarning: Input type into Linear4bit is torch.float16, but bnb_4bit_compute_dtype=torch.float32 (default). This will lead to slow inference or training speed.
  warnings.warn(
2024-09-27 15:20:21.306457: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-09-27 15:20:21.318747: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-09-27 15:20:21.339370: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to reg

Answer: CTLA4-Ig is a medication that is used to treat autoimmune diseases, such as rheumatoid arthritis, psoriasis, and lupus. It is also used to treat Crohn's disease and ulcerative colitis.


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Answer: Camptocormia is a rare movement disorder characterized by a rigid, flexed posture of the trunk and arms. The exact cause of camptocormia is still unknown, but it is often associated with neurological disorders such as Parkinson's disease, stroke, and neurodegenerative diseases. 

The examination of camptocormia typically involves a detailed clinical evaluation and a thorough medical history. The examination may include:

1. Observation of posture: A physician may observe the patient's posture to assess the degree of camptocormia. This includes noting the degree of flexion and rotation of the trunk and arms.

2. Motor examination: The physician may perform a motor examination to assess muscle strength, tone, and coordination. This may include testing muscle strength in the arms and legs, as well as assessing coordination and balance.

3. Reflex examination: The physician may also perform a reflex examination to assess the patient's reflexes, which may be abnormal in camptocormia

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


...

Answer: The only second-row transition metal essential for biological systems is Manganese (Mn).


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Answer: A key component of Enhanced Recovery After Surgery (ERAS) protocols is the emphasis on early mobilization and removal of restrictions on patient activity. This is often referred to as "fast track" or "fast track anesthesia" and is aimed at reducing perioperative complications and improving patient outcomes.


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Answer: The only opportunity to obtain diagnostic tissue to diagnose and subclassify Renal Cell Carcinoma (RCC) is through surgical resection. This may include radical nephrectomy, partial nephrectomy, or a biopsy. Biopsy is not always necessary for diagnosis, but a tissue diagnosis is necessary for subclassification and staging of RCC.
Answer: The onset of arthritis in a mouse model is a complex process that involves a combination of genetic and environmental factors. The exact onset of arthritis in mice is still not well understood and is an active area of research. However, there are several factors that have been identified to play a role in the onset of arthritis in mice.


In [12]:
import pandas as pd

df_to_csv = pd.DataFrame({'question': questions_list,
                   'response': answers_list})

df_to_csv.to_csv(response_filename)

: 